> Experiments with [xsa.access](../trapallada/xsa/access.py) module

In [ ]:
import pathlib
import os
import tempfile


import trapallada.xsa.access
import trapallada.xsa.util
import trapallada.plot

from astroquery.esa.xmm_newton import XMMNewton

So that the notebook is aware of modifications in the the `import`ed modules

In [ ]:
%load_ext autoreload
%autoreload 2

All the available tables (only a few are printed)

In [ ]:
tables = trapallada.xsa.access.get_all_tables()
tables[:5]

Get *metadata* for observations whose duration is in a given `duration_interval`

In [ ]:
obs_metadata_df = trapallada.xsa.access.get_observations_in_interval(duration_interval=(35_000, 45_000))
obs_metadata_df

In order to locate a particular observation

In [ ]:
# obs = '0200510101' # duration not in the requested interval!!
# obs = '0006010401'
obs = '0720750201'
obs_metadata_df[obs_metadata_df['observation_id'] == obs]

In [ ]:
with tempfile.TemporaryDirectory(delete=True) as tmp_dir:

    # context's descriptor is turned into a `Pathlib`
    tmp_dir_root = pathlib.Path(tmp_dir)

    tar_file = trapallada.xsa.access.download_obs(obs, output_directory=tmp_dir_root)
    
    obs_df, header_df = trapallada.xsa.util.read_tar(tar_file, max_n_sources=4)

Duration is

In [ ]:
duration = obs_df.index[-1] - obs_df.index[0]
duration, duration.total_seconds()

The directory where observations' data will be saved

In [ ]:
output_directory =  pathlib.Path(os.environ['MY_DATA_DIR']) / 'trapallada'
output_directory.mkdir(exist_ok=True)

Let us select a few observations to download

In [ ]:
obs_to_download = obs_metadata_df['observation_id'].iloc[:10]
obs_to_download

Downloading the first observation into a temporary directory

In [ ]:
with tempfile.TemporaryDirectory(delete=False) as tmp_dir:

    # context's descriptor is turned into a `Pathlib`
    tmp_dir_root = pathlib.Path(tmp_dir)

    tar_file = trapallada.xsa.access.download_obs(obs_to_download[0], output_directory=tmp_dir_root)
print(tar_file)

In [ ]:
# from astropy.table import Table
# tar_file
# trapallada.xsa.util.read_fits_file(tar_file)

# Low-level calls to `XMMNewton`

## `v_instrument_mode` table

*Instrument mode* table

In [ ]:
instrument_mode_table = 'v_instrument_mode'
XMMNewton.get_columns('xsa.' + instrument_mode_table)

A sample

In [ ]:
table = XMMNewton.query_xsa_tap(
    f'select TOP 20 * from {instrument_mode_table}',
)
table.to_pandas()

All the columns in the `v_all_observations`

In [ ]:
XMMNewton.get_columns('xsa.v_all_observations') 